In [2]:
import os  # Lets us set the Java location before Spark starts.
import shutil  # Finds Ubuntu's Java command.
from pathlib import Path  # Helps us build filesystem paths.
from pyspark.sql import SparkSession  # Creates our Spark session.
from delta import configure_spark_with_delta_pip  # Adds Delta's Java package.

java_command = shutil.which("java")  # Find Java in the WSL environment.
if java_command is None:  # Check that Java was found.
    raise RuntimeError("Java was not found in WSL")  # Stop with a clear message if it was not.

java_home = Path(java_command).resolve().parent.parent  # Find the Java installation folder.
os.environ["JAVA_HOME"] = str(java_home)  # Point Spark to Ubuntu's Java.

project_root = Path("/mnt/f/Big Data/Retail_Lakehouse_Learning")  # Locate our project.

builder = (  # Define Spark's settings before it starts.
    SparkSession.builder
    .master("local[2]")  # Use two processing threads on this computer.
    .appName("RetailSilverProducts")  # Name this Silver job.
    .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension")  # Enable Delta features.
    .config("spark.sql.catalog.spark_catalog", "org.apache.spark.sql.delta.catalog.DeltaCatalog")  # Enable Delta tables.
)

spark = configure_spark_with_delta_pip(builder).getOrCreate()  # Start Spark with Delta support.

print("Project found:", project_root.is_dir())  # Confirm the project path works.
print("Spark version:", spark.version)  # Confirm Spark started.

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/10/09 19:18:55 WARN Utils: Your hostname, USMAN-ALI, resolves to a loopback address: 127.0.1.1; using 10.255.255.254 instead (on interface lo)
26/10/09 19:18:55 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
:: loading settings :: url = jar:file:/home/usmanali611b/.venvs/retail-lakehouse/lib/python3.12/site-packages/pyspark/jars/ivy-2.5.3.jar!/org/apache/ivy/core/settings/ivysettings.xml
Ivy Default Cache set to: /home/usmanali611b/.ivy2.5.2/cache
The jars for the packages stored in: /home/usmanali611b/.ivy2.5.2/jars
io.delta#delta-spark_4.1_2.13 added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-4057d65e-5d2b-4d04-b429-7ae680579806;1.0
	confs: [default]
	found io.delta#delta-spark_4.1_2.13;4.3.1 in central
	found io.delta#delta-storage;4.3.1 in central
	found io.unitycatalog#unitycatalog-client;0.5.0 in central
	found org.slf4j#slf4j-api;2.0.1

Project found: True
Spark version: 4.1.1


In [4]:
bronze_products_path = project_root / "lakehouse" / "bronze" / "data" / "products"  # Locate the source-shaped products table.
silver_suppliers_path = project_root / "lakehouse" / "silver" / "data" / "suppliers"  # Locate cleaned suppliers.

products_bronze_df = spark.read.format("delta").load(str(bronze_products_path))  # Read Bronze products.
suppliers_silver_df = spark.read.format("delta").load(str(silver_suppliers_path))  # Read Silver suppliers.

print("Bronze products:", products_bronze_df.count())  # Check the product source count.
print("Silver suppliers:", suppliers_silver_df.count())  # Check the supplier reference count.
products_bronze_df.printSchema()  # Inspect product column names and types.
products_bronze_df.show(5, truncate=False)  # Preview five source-shaped products.

26/10/09 19:19:42 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.
                                                                                

Bronze products: 120


Silver suppliers: 18
root
 |-- product_id: string (nullable = true)
 |-- product_name: string (nullable = true)
 |-- category: string (nullable = true)
 |-- supplier_id: string (nullable = true)
 |-- cost_price: string (nullable = true)
 |-- list_price: string (nullable = true)
 |-- currency_code: string (nullable = true)
 |-- active_flag: string (nullable = true)



[Stage 19:>                                                         (0 + 1) / 1]

+----------+-----------------------------+---------+-----------+----------+----------+-------------+-----------+
|product_id|product_name                 |category |supplier_id|cost_price|list_price|currency_code|active_flag|
+----------+-----------------------------+---------+-----------+----------+----------+-------------+-----------+
|P0001     |Ever Mineral Water Standard  |Beverages|SUP010     |433.39    |660       |PKR          |True       |
|P0002     |Nova Mineral Water Family    |Beverages|SUP009     |182.14    |265       |PKR          |True       |
|P0003     |FreshCo Orange Juice Standard|Beverages|SUP009     |664.17    |1000      |PKR          |True       |
|P0004     |FreshCo Orange Juice Family  |Beverages|SUP017     |1010.69   |1410      |PKR          |True       |
|P0005     |FreshCo Mango Juice Standard |Beverages|SUP005     |960.41    |1405      |PKR          |True       |
+----------+-----------------------------+---------+-----------+----------+----------+----------

In [5]:
from pyspark.sql import functions as F  # Import Spark functions for column checks.

products_with_clean_id_df = products_bronze_df.withColumn(  # Add the ID form Silver would use.
    "clean_product_id", F.trim(F.col("product_id"))  # Remove spaces around product IDs.
)

missing_id_count = products_with_clean_id_df.filter(  # Find products without a usable ID.
    F.col("clean_product_id").isNull()  # Include null IDs.
    | (F.col("clean_product_id") == "")  # Include empty IDs after trimming.
).count()  # Count those rows.

duplicate_id_groups = (  # Count product IDs repeated after trimming.
    products_with_clean_id_df
    .groupBy("clean_product_id")  # Group rows sharing one cleaned ID.
    .count()  # Count rows in each group.
    .filter(F.col("count") > 1)  # Keep duplicate groups.
    .count()  # Count those groups.
)

print("Missing product IDs:", missing_id_count)  # Show missing keys.
print("Duplicate cleaned product IDs:", duplicate_id_groups)  # Show duplicate keys.

products_bronze_df.groupBy("category").count().orderBy("category").show(truncate=False)  # List observed categories.
products_bronze_df.groupBy("currency_code").count().orderBy("currency_code").show()  # List observed currencies.
products_bronze_df.groupBy("active_flag").count().orderBy("active_flag").show()  # List observed flag values.

Missing product IDs: 0
Duplicate cleaned product IDs: 0
+-------------+-----+
|category     |count|
+-------------+-----+
|Bakery       |20   |
|Beverages    |20   |
|Grocery      |20   |
|Household    |20   |
|Personal Care|20   |
|Snacks       |20   |
+-------------+-----+



+-------------+-----+
|currency_code|count|
+-------------+-----+
|          PKR|  120|
+-------------+-----+



+-----------+-----+
|active_flag|count|
+-----------+-----+
|       True|  120|
+-----------+-----+



In [6]:
products_clean_df = products_bronze_df.select(  # Create a new DataFrame; leave Bronze unchanged.
    F.trim(F.col("product_id")).alias("product_id"),  # Clean the product key.
    F.trim(F.col("product_name")).alias("product_name"),  # Clean the display name.
    F.trim(F.col("category")).alias("category"),  # Clean the category text.
    F.trim(F.col("supplier_id")).alias("supplier_id"),  # Clean the supplier reference.
    F.trim(F.col("cost_price")).try_cast("decimal(12,2)").alias("cost_price"),  # Convert cost to an exact two-decimal value.
    F.trim(F.col("list_price")).try_cast("decimal(12,2)").alias("list_price"),  # Convert retail price the same way.
    F.trim(F.col("currency_code")).alias("currency_code"),  # Clean the currency code.
    F.trim(F.col("active_flag")).try_cast("boolean").alias("active_flag"),  # Convert the text flag to boolean.
    F.to_json(F.struct(*[F.col(name) for name in products_bronze_df.columns])).alias("source_record"),  # Preserve original CSV values for error evidence.
)

print("Cleaned product rows:", products_clean_df.count())  # Confirm cleaning retained all source rows.
products_clean_df.printSchema()  # Inspect the converted price and flag types.
products_clean_df.show(5, truncate=False)  # Preview converted values and the original record.

Cleaned product rows: 120
root
 |-- product_id: string (nullable = true)
 |-- product_name: string (nullable = true)
 |-- category: string (nullable = true)
 |-- supplier_id: string (nullable = true)
 |-- cost_price: decimal(12,2) (nullable = true)
 |-- list_price: decimal(12,2) (nullable = true)
 |-- currency_code: string (nullable = true)
 |-- active_flag: boolean (nullable = true)
 |-- source_record: string (nullable = true)



+----------+-----------------------------+---------+-----------+----------+----------+-------------+-----------+--------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+
|product_id|product_name                 |category |supplier_id|cost_price|list_price|currency_code|active_flag|source_record                                                                                                                                                                                           |
+----------+-----------------------------+---------+-----------+----------+----------+-------------+-----------+--------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+
|P0001     |Ever Mineral Water Standard  |Beverages|SUP010

In [7]:
supplier_keys_df = (  # Make a small reference DataFrame for the join.
    suppliers_silver_df
    .select("supplier_id")  # Keep only the supplier key.
    .withColumn("supplier_found", F.lit(True))  # Mark a matching supplier.
)

products_with_supplier_df = products_clean_df.join(  # Attach a match marker to each product.
    supplier_keys_df,  # Use cleaned Silver suppliers as the reference.
    on="supplier_id",  # Match the supplier_id column in both tables.
    how="left",  # Keep every product, including those without a match.
)

joined_count = products_with_supplier_df.count()  # Check how many rows the join produced.
if joined_count != products_clean_df.count():  # A reference join must not multiply product rows.
    raise ValueError("Supplier join changed the product row count")  # Stop if supplier keys are duplicated.

unknown_supplier_count = products_with_supplier_df.filter(  # Find products without a supplier match.
    F.col("supplier_found").isNull()  # The marker is null when the left join found nothing.
).count()  # Count those products.

print("Products after supplier join:", joined_count)  # Expect 120.
print("Products with unknown supplier:", unknown_supplier_count)  # Expect 0.

Products after supplier join: 120
Products with unknown supplier: 0


In [8]:
allowed_categories = (  # Provisional categories observed in this training source.
    "Bakery", "Beverages", "Grocery",  # First three category names.
    "Household", "Personal Care", "Snacks",  # Remaining category names.
)

product_checks = {  # Give each check a readable output label.
    "Missing product name": F.col("product_name").isNull() | (F.col("product_name") == ""),  # Require a name.
    "Missing or unlisted category": F.col("category").isNull() | (F.col("category") == "") | ~F.col("category").isin(*allowed_categories),  # Check the provisional list.
    "Missing or invalid cost": F.col("cost_price").isNull() | (F.col("cost_price") < 0),  # Require a nonnegative decimal.
    "Missing or negative list price": F.col("list_price").isNull() | (F.col("list_price") < 0),  # Require a nonnegative decimal.
    "Missing or wrong currency": F.col("currency_code").isNull() | (F.col("currency_code") != "PKR"),  # Require PKR.
    "Missing or invalid active flag": F.col("active_flag").isNull(),  # Catch an absent or unconvertible flag.
}

for label, condition in product_checks.items():  # Run each check against cleaned products.
    issue_count = products_with_supplier_df.filter(condition).count()  # Count matching rows.
    print(f"{label}: {issue_count}")  # Show this check's result.

below_cost_count = products_with_supplier_df.filter(  # Find the price-comparison warning.
    F.col("cost_price").isNotNull()  # Require a converted cost for comparison.
    & F.col("list_price").isNotNull()  # Require a converted list price.
    & (F.col("list_price") < F.col("cost_price"))  # Find list prices below cost.
).count()  # Count warning rows.

print("List price below cost (warning):", below_cost_count)  # Report separately from rejection checks.

Missing product name: 0
Missing or unlisted category: 0
Missing or invalid cost: 0
Missing or negative list price: 0
Missing or wrong currency: 0
Missing or invalid active flag: 0
List price below cost (warning): 0


In [9]:
from pyspark.sql.window import Window  # Lets us count products sharing one cleaned ID.

def missing_product_text(column_name):  # Reuse one check for required text fields.
    column = F.col(column_name)  # Select the named column.
    return column.isNull() | (column == "")  # Flag null or empty text after trimming.

products_checked_df = products_with_supplier_df.withColumn(  # Add a count for each product ID.
    "id_occurrences",  # Name the temporary count column.
    F.count(F.lit(1)).over(Window.partitionBy("product_id")),  # Count rows with this cleaned ID.
).withColumn(  # Add reasons that make a product invalid.
    "quality_issue",  # Store all applicable rejection reasons.
    F.concat_ws(  # Join issue names while skipping checks that did not fail.
        "; ",  # Separate multiple reasons.
        F.when(missing_product_text("product_id"), "missing_product_id"),  # Require a product key.
        F.when(F.col("id_occurrences") > 1, "duplicate_product_id"),  # Reject duplicate cleaned keys.
        F.when(missing_product_text("product_name"), "missing_product_name"),  # Require a name.
        F.when(missing_product_text("category"), "missing_category"),  # Require a category.
        F.when(~missing_product_text("category") & ~F.col("category").isin(*allowed_categories), "unlisted_category"),  # Apply our provisional category list.
        F.when(missing_product_text("supplier_id"), "missing_supplier_id"),  # Require a supplier reference.
        F.when(~missing_product_text("supplier_id") & F.col("supplier_found").isNull(), "unknown_supplier_id"),  # Require a Silver supplier match.
        F.when(F.col("cost_price").isNull(), "missing_or_invalid_cost"),  # Catch absent or unconvertible costs.
        F.when(F.col("cost_price") < 0, "negative_cost"),  # Reject negative costs.
        F.when(F.col("list_price").isNull(), "missing_or_invalid_list_price"),  # Catch absent or unconvertible prices.
        F.when(F.col("list_price") < 0, "negative_list_price"),  # Reject negative prices.
        F.when(missing_product_text("currency_code"), "missing_currency"),  # Require a currency.
        F.when(~missing_product_text("currency_code") & (F.col("currency_code") != "PKR"), "wrong_currency"),  # Require PKR.
        F.when(F.col("active_flag").isNull(), "missing_or_invalid_active_flag"),  # Require a valid boolean.
    ),
).withColumn(  # Record the separate price warning.
    "warning_issue",  # Name the warning column.
    F.when(  # Add a warning only when both prices are valid and nonnegative.
        (F.col("cost_price") >= 0)  # Check the cost can be compared.
        & (F.col("list_price") >= 0)  # Check the list price can be compared.
        & (F.col("list_price") < F.col("cost_price")),  # Find a below-cost list price.
        "list_price_below_cost",  # Describe the warning.
    ).otherwise(""),  # Use empty text when there is no warning.
)

accepted_count = products_checked_df.filter(F.col("quality_issue") == "").count()  # Count valid products.
rejected_count = products_checked_df.filter(F.col("quality_issue") != "").count()  # Count invalid products.
warning_count = products_checked_df.filter(  # Count warnings among accepted products.
    (F.col("quality_issue") == "") & (F.col("warning_issue") != "")  # Keep valid rows with a warning.
).count()  # Count them.

print("Products ready for Silver:", accepted_count)  # Show valid rows.
print("Products needing rejection:", rejected_count)  # Show invalid rows.
print("Accepted products with warning:", warning_count)  # Show warnings separately.
print("All rows accounted for:", accepted_count + rejected_count == 120)  # Check the split.

Products ready for Silver: 120
Products needing rejection: 0
Accepted products with warning: 0
All rows accounted for: True


In [11]:
silver_products_df = (  # Build the DataFrame for the Silver table.
    products_checked_df
    .filter(F.col("quality_issue") == "")  # Keep products that passed rejection rules.
    .drop("id_occurrences", "quality_issue", "supplier_found", "source_record")  # Remove temporary checks and raw evidence.
        .select(  # Choose a clear column order for the saved Silver table.
        "product_id",  # Product primary key.
        "product_name",  # Product display name.
        "category",  # Product category.
        "supplier_id",  # Reference to Silver suppliers.
        "cost_price",  # Typed cost.
        "list_price",  # Typed retail price.
        "currency_code",  # Currency.
        "active_flag",  # Typed active flag.
        "warning_issue",  # Non-rejecting price warning.
    )
)

rejected_products_df = (  # Build the DataFrame for error evidence.
    products_checked_df
    .filter(F.col("quality_issue") != "")  # Keep products that failed a rejection rule.
    .drop("id_occurrences", "supplier_found")  # Keep quality_issue and source_record for investigation.
)

print("Silver product candidates:", silver_products_df.count())  # Expect 120.
print("Rejected product candidates:", rejected_products_df.count())  # Expect 0.
print("Silver columns:", silver_products_df.columns)  # Confirm business fields plus warning_issue.

Silver product candidates: 120
Rejected product candidates: 0
Silver columns: ['product_id', 'product_name', 'category', 'supplier_id', 'cost_price', 'list_price', 'currency_code', 'active_flag', 'warning_issue']


In [12]:
from uuid import uuid4  # Create a unique ID for this processing run.

bronze_count = products_bronze_df.count()  # Count input products.
silver_count = silver_products_df.count()  # Count accepted products.
rejected_count = rejected_products_df.count()  # Count rejected products.

if silver_count + rejected_count != bronze_count:  # Check every product has an outcome.
    raise ValueError("Product rows were lost during Silver processing")  # Stop before writing.

run_id = str(uuid4())  # Identify this processing run.
saved_rejected_count = 0  # Start with no saved rejections.

if rejected_count > 0:  # Save error evidence only when rejected products exist.
    error_path = project_root / "error" / "data" / "products"  # Choose the ignored error folder.
    error_path.parent.mkdir(parents=True, exist_ok=True)  # Ensure its parent exists.

    errors_to_save_df = (  # Add audit details to the rejected rows.
        rejected_products_df
        .withColumn("run_id", F.lit(run_id))  # Label this run's errors.
        .withColumn("rejected_at", F.current_timestamp())  # Record processing time.
    )

    errors_to_save_df.write.format("delta").mode("append").save(str(error_path))  # Preserve earlier error records.

    saved_rejected_count = (  # Verify this run's error rows.
        spark.read.format("delta").load(str(error_path))  # Read the error table.
        .filter(F.col("run_id") == run_id)  # Select only this run.
        .count()  # Count saved rejections.
    )

    if saved_rejected_count != rejected_count:  # Compare saved and expected errors.
        raise ValueError("Saved product rejection count does not match")  # Stop before Silver changes.

silver_products_path = project_root / "lakehouse" / "silver" / "data" / "products"  # Choose the Silver Delta folder.
silver_products_path.parent.mkdir(parents=True, exist_ok=True)  # Ensure its parent exists.

silver_products_df.write.format("delta").mode("overwrite").save(str(silver_products_path))  # Save the clean snapshot.

saved_products_df = spark.read.format("delta").load(str(silver_products_path))  # Read Silver back.
saved_count = saved_products_df.count()  # Count saved products.
saved_warning_count = saved_products_df.filter(F.col("warning_issue") != "").count()  # Count saved warnings.

if saved_count != silver_count:  # Compare saved rows with accepted rows.
    raise ValueError("Saved Silver product count does not match")  # Stop if rows differ.

print("Bronze products:", bronze_count)  # Show source count.
print("Saved Silver products:", saved_count)  # Show saved count.
print("Rejected products this run:", saved_rejected_count)  # Show error count.
print("Saved products with warnings:", saved_warning_count)  # Show warning count.
print("Delta log exists:", (silver_products_path / "_delta_log").is_dir())  # Confirm Delta output.

Bronze products: 120
Saved Silver products: 120
Rejected products this run: 0
Saved products with warnings: 0
Delta log exists: True


In [13]:
saved_types = {  # Build a lookup from saved column name to Spark type.
    field.name: field.dataType.simpleString()  # Record one saved field's type.
    for field in saved_products_df.schema.fields  # Inspect every saved field.
}

print("Saved columns:", saved_products_df.columns)  # Confirm the chosen column order.
print("Cost type:", saved_types["cost_price"])  # Expect decimal(12,2).
print("List price type:", saved_types["list_price"])  # Expect decimal(12,2).
print("Active flag type:", saved_types["active_flag"])  # Expect boolean.

if saved_types["cost_price"] != "decimal(12,2)":  # Check the saved cost conversion.
    raise ValueError("Silver product cost has the wrong type")  # Stop if incorrect.

if saved_types["list_price"] != "decimal(12,2)":  # Check the saved list-price conversion.
    raise ValueError("Silver product list price has the wrong type")  # Stop if incorrect.

if saved_types["active_flag"] != "boolean":  # Check the saved flag conversion.
    raise ValueError("Silver product active flag has the wrong type")  # Stop if incorrect.

Saved columns: ['product_id', 'product_name', 'category', 'supplier_id', 'cost_price', 'list_price', 'currency_code', 'active_flag', 'warning_issue']
Cost type: decimal(12,2)
List price type: decimal(12,2)
Active flag type: boolean
